# Loan Default & Credit Risk Analysis

Mini-project covering data loading, cleaning, analysis, 15 visualizations, and Power BI preparation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../Dataset/Loan_Credit_Risk_Dataset.csv")
df.head()

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())
display(df.tail())
df.info()
display(df.describe(include="all").T)

In [ ]:
# Missing values and duplicates
print(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())

# Cleaning
df["Education"] = df["Education"].fillna(df["Education"].mode()[0])
df["Employment_Status"] = df["Employment_Status"].fillna(df["Employment_Status"].mode()[0])
df["Annual_Income"] = pd.to_numeric(df["Annual_Income"], errors="coerce").fillna(df["Annual_Income"].median())
df["Credit_Score"] = pd.to_numeric(df["Credit_Score"], errors="coerce").fillna(df["Credit_Score"].median())
df = df.drop_duplicates().reset_index(drop=True)

df["Credit_Score_Group"] = pd.cut(
    df["Credit_Score"], [-1,549,649,699,850],
    labels=["Poor (<550)","Fair (550-649)","Good (650-699)","Excellent (700+)"]
)
df["Income_Group"] = pd.qcut(df["Annual_Income"], 4, labels=["Low","Lower-Mid","Upper-Mid","High"], duplicates="drop")
df.to_csv("../Dataset/Loan_Credit_Risk_Cleaned.csv", index=False)
df.isna().sum()

## Required Analysis

In [ ]:
total_customers = len(df)
total_loans = df["Loan_ID"].nunique()
defaulted = (df["Default_Status"]=="Default").sum()
non_defaulted = (df["Default_Status"]=="Non-Default").sum()
default_rate = defaulted / total_loans * 100

print("Total Customers:", total_customers)
print("Total Loans:", total_loans)
print("Defaulted Loans:", defaulted)
print("Non-Defaulted Loans:", non_defaulted)
print("Overall Default Rate:", round(default_rate,2), "%")
print("Average Annual Income:", round(df["Annual_Income"].mean(),2))
print("Average Credit Score:", round(df["Credit_Score"].mean(),2))
print("Average Loan Amount:", round(df["Loan_Amount"].mean(),2))
print("Average Interest Rate:", round(df["Interest_Rate"].mean(),2))
print("Average DTI:", round(df["Debt_to_Income_Ratio"].mean(),2))

In [ ]:
def default_rate(col):
    return (df.groupby(col)["Default_Status"]
              .apply(lambda s: (s=="Default").mean()*100)
              .sort_values(ascending=False))

for c in ["Loan_Type","Employment_Status","Education","Property_Ownership","Credit_Score_Group","Income_Group","Region"]:
    print("\n", c)
    display(default_rate(c))

print("\nCredit score vs default")
display(df.groupby("Default_Status")["Credit_Score"].mean())

print("\nIncome vs default")
display(df.groupby("Default_Status")["Annual_Income"].mean())

print("\nLoan amount vs default")
display(df.groupby("Default_Status")["Loan_Amount"].mean())

print("\nPrevious defaults vs current default")
display(default_rate("Previous_Defaults"))

print("\nTop 10 loans by amount")
display(df.nlargest(10, "Loan_Amount")[["Loan_ID","Customer_ID","Loan_Type","Loan_Amount","Credit_Score","Default_Status"]])

## 15 Required Visualizations
The notebook can generate the same 15 charts saved in the `Visualizations` folder.

In [ ]:
# Example visualization code; repeat/adapt for all required charts.
fig, ax = plt.subplots()
df["Default_Status"].value_counts().plot.pie(autopct="%1.1f%%", ax=ax)
ax.set_ylabel("")
ax.set_title("Loan Default Distribution")
plt.show()

## Business Insights
Use the calculated default rates, group comparisons, correlations, and charts to write 5–10 business insights.